In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification


train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


Prepare tokenizer and models

In [3]:
deberta_tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

deberta_model = AutoModelForSequenceClassification.from_pretrained("microsoft/deberta-v3-small", num_labels=5)

roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")

roberta_model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deberta_model.to(device)
roberta_model.to(device)

deberta_model.eval()
roberta_model.eval()


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

(answer format : eg - A, probability of A)


In [4]:
labels = ["A","B","C","D","E"]

row = train.iloc[25]

text = row["prompt"]

for option in labels:
    text = text + " [SEP] " + row[option]

In [5]:
# prob using deberta

encoding = deberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

with torch.no_grad():

    output = deberta_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

deberta_prob = torch.softmax(output.logits, dim=1)

print(deberta_prob)

tensor([[0.2595, 0.1899, 0.1891, 0.1958, 0.1656]], device='cuda:0',
       dtype=torch.float16)


In [6]:
best = torch.argmax(deberta_prob).item()

answer = labels[best]

confidence = deberta_prob[0][best].item()

print("Answer 1")
print("Predicted Answer :", answer)
print("Confidence :", confidence)

Answer 1
Predicted Answer : A
Confidence : 0.259521484375


In [7]:
# prob using roberta

encoding = roberta_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

with torch.no_grad():

    output = roberta_model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

roberta_prob = torch.softmax(output.logits, dim=1)

print(roberta_prob)

tensor([[0.1890, 0.1828, 0.2196, 0.2040, 0.2047]], device='cuda:0')


Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?



In [8]:
average_prob = (deberta_prob + roberta_prob) / 2

best = torch.argmax(average_prob).item()

answer = labels[best]

confidence = average_prob[0][best].item()

print("Answer 2")
print("Answer :", answer)
print("Probability :", confidence)

Answer 2
Answer : A
Probability : 0.22425037622451782


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?



In [9]:
ensemble_prob = (0.7 * deberta_prob) + (0.3 * roberta_prob)

best = torch.argmax(ensemble_prob).item()

answer = labels[best]

confidence = ensemble_prob[0][best].item()

print("Answer 3")
print("Answer :", answer)
print("Probability :", confidence)

Answer 3
Answer : A
Probability : 0.23833441734313965


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

*
Example : C A E



In [10]:
indices = torch.argsort(ensemble_prob, descending=True)

top3 = []

for i in indices[0][:3]:
    top3.append(labels[i])

prediction = " ".join(top3)

print("Answer 4")
print(prediction)

Answer 4
A D C


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)?



In [11]:
deberta_probabilities = []
roberta_probabilities = []
ensemble_probabilities = []

predictions = []

for i in range(len(test)):

    row = test.iloc[i]

    text = row["prompt"]

    for option in labels:
        text += " [SEP] " + row[option]

    # deberta

    encoding = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        output = deberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    deberta_prob = torch.softmax(output.logits, dim=1)

    # roberta

    encoding = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        output = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    roberta_prob = torch.softmax(output.logits, dim=1)

    # ensemble

    ensemble_prob = (0.7 * deberta_prob) + (0.3 * roberta_prob)

    deberta_probabilities.append(deberta_prob)
    roberta_probabilities.append(roberta_prob)
    ensemble_probabilities.append(ensemble_prob)

    indices = torch.argsort(ensemble_prob, descending=True)

    top3 = []

    for j in indices[0][:3]:
        top3.append(labels[j])

    predictions.append(" ".join(top3))

In [12]:
submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

submission["Prediction"] = predictions

submission.to_csv("submission.csv", index=False)

print("submission file created!")

print("Answer 5")
print("Number of prediction rows :", len(submission))
submission.head()

submission file created!
Answer 5
Number of prediction rows : 500


,ID,Prediction
0,1,A C B
1,2,D A C
2,3,A C B
3,4,A D C
4,5,A D B


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?



In [13]:
changed_predictions = 0

for i in range(50):

    row = test.iloc[i]

    # Original Prompt 

    original_text = row["prompt"]

    for option in labels:
        original_text += " [SEP] " + row[option]

    # augmented prompt 

    augmented_text = "Answer the following multiple-choice question carefully: " + row["prompt"]

    for option in labels:
        augmented_text += " [SEP] " + row[option]

    # original prediction 

    encoding = deberta_tokenizer(
        original_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        output = deberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    original_prob = torch.softmax(output.logits, dim=1)

    # augmented prediction 

    encoding = deberta_tokenizer(
        augmented_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        output = deberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    augmented_prob = torch.softmax(output.logits, dim=1)

    # Average 

    tta_prob = (original_prob + augmented_prob) / 2

    original_answer = torch.argmax(original_prob).item()

    tta_answer = torch.argmax(tta_prob).item()

    if original_answer != tta_answer:
        changed_predictions += 1

In [14]:
print("Answer 6")
print("Number of changed Top-1 predictions:", changed_predictions)

Answer 6
Number of changed Top-1 predictions: 7


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?



In [15]:
deberta_top1 = []
ensemble_top1 = []

deberta_confidence = []
ensemble_confidence = []

deberta_top3 = []
ensemble_top3 = []

for i in range(100):

    row = test.iloc[i]

    text = row["prompt"]

    for option in labels:
        text += " [SEP] " + row[option]

    # deberta

    encoding = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        output = deberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    deberta_prob = torch.softmax(output.logits, dim=1)

    # roberta

    encoding = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():

        output = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    roberta_prob = torch.softmax(output.logits, dim=1)

    # Ensemble

    ensemble_prob = (0.7 * deberta_prob) + (0.3 * roberta_prob)

    # Top-1 

    deberta_best = torch.argmax(deberta_prob).item()
    ensemble_best = torch.argmax(ensemble_prob).item()

    deberta_top1.append(labels[deberta_best])
    ensemble_top1.append(labels[ensemble_best])

    # Confidence 

    deberta_confidence.append(torch.max(deberta_prob).item())
    ensemble_confidence.append(torch.max(ensemble_prob).item())

    # Top-3 

    deberta_indices = torch.argsort(
        deberta_prob,
        descending=True
    )

    ensemble_indices = torch.argsort(
        ensemble_prob,
        descending=True
    )

    top3 = []

    for j in deberta_indices[0][:3]:
        top3.append(labels[j])

    deberta_top3.append(" ".join(top3))

    top3 = []

    for j in ensemble_indices[0][:3]:
        top3.append(labels[j])

    ensemble_top3.append(" ".join(top3))

In [16]:
count = 0

for i in range(100):

    if deberta_top1[i] != ensemble_top1[i]:
        count += 1

print("Answer 7:", count)

Answer 7: 2


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?



In [17]:
count = 0

for i in range(100):

    gain = ensemble_confidence[i] - deberta_confidence[i]

    if gain > 0:
        count += 1

print("Answer 8") 
print(count)

Answer 8
0


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

*
Examples:

A C D vs. A D C



In [18]:
count = 0

for i in range(100):

    if deberta_top3[i] != ensemble_top3[i]:
        count += 1

print("Answer 9")
print(count)

Answer 9
40


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score? 

*
(Round to 4 decimal places.)


In [19]:
def map_at_3(actual, predicted):

    score = 0

    for i in range(len(actual)):

        prediction = predicted[i].split()

        if actual[i] in prediction:

            rank = prediction.index(actual[i]) + 1

            score += 1 / rank

    return score / len(actual)

In [20]:
ensemble_top3 = []

for i in range(100):

    # Running on training data since test data has no answer label
    row = train.iloc[i]

    text = row["prompt"]

    for option in labels:
        text += " [SEP] " + row[option]

    # deberta

    encoding = deberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        output = deberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    deberta_prob = torch.softmax(output.logits, dim=1)

    # roberta

    encoding = roberta_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        output = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    roberta_prob = torch.softmax(output.logits, dim=1)

    # Ensemble

    ensemble_prob = (0.7 * deberta_prob) + (0.3 * roberta_prob)

    ensemble_indices = torch.argsort(
        ensemble_prob,
        descending=True
    )

    top3 = []

    for j in ensemble_indices[0][:3]:
        top3.append(labels[j])

    ensemble_top3.append(" ".join(top3))

In [21]:
actual = train.iloc[:100]["answer"]

score = map_at_3(actual, ensemble_top3)

print("Answer 10")
print(round(score, 4))

Answer 10
0.3733
